# Beca 18 RAG Chatbot
Retrieval-Augmented Generation pipeline to answer questions about Peru's Beca 18 scholarship regulations (PRONABEC).

## Step 0 — Environment Setup
Load the Gemini API key from `.env` and print all relevant package versions.

In [1]:
import subprocess
subprocess.run(["pip", "install", "-r", "../requirements.txt", "-q"], check=True)

import os
from dotenv import load_dotenv, find_dotenv
from importlib.metadata import version

load_dotenv(find_dotenv())

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
assert GEMINI_API_KEY, "GEMINI_API_KEY not found — check your .env file"
print("API key loaded successfully.")

# Print package versions
import pypdf, tiktoken, chromadb, ipywidgets, tqdm
import google.genai as genai

packages = {
    "pypdf": pypdf.__version__,
    "tiktoken": tiktoken.__version__,
    "langchain-text-splitters": version("langchain-text-splitters"),
    "google-genai": genai.__version__,
    "chromadb": chromadb.__version__,
    "ipywidgets": ipywidgets.__version__,
    "tqdm": tqdm.__version__,
}

print("\nPackage versions:")
for pkg, ver in packages.items():
    print(f"  {pkg}: {ver}")

API key loaded successfully.

Package versions:
  pypdf: 6.11.0
  tiktoken: 0.13.0
  langchain-text-splitters: 1.1.2
  google-genai: 2.3.0
  chromadb: 1.5.9
  ipywidgets: 8.1.7
  tqdm: 4.67.1


## Step 1 — PDF Text Extraction
Extract text page-by-page using `pypdf`, insert `[PAGE N]` markers, and clean the raw text.

In [2]:
import re
import pypdf

PDF_PATH = "../data/beca18_reglamento.pdf"

def extract_text(pdf_path: str) -> str:
    reader = pypdf.PdfReader(pdf_path)
    pages = []
    for i, page in enumerate(reader.pages, start=1):
        raw = page.extract_text() or ""
        # Insert page marker
        text = f"[PAGE {i}]\n{raw}"
        pages.append(text)
    return "\n".join(pages)

def clean_text(text: str) -> str:
    # Remove line breaks inside sentences (keep paragraph breaks)
    text = re.sub(r"(?<!\n)\n(?!\n)", " ", text)
    # Collapse multiple spaces
    text = re.sub(r" {2,}", " ", text)
    # Strip leading/trailing whitespace per line
    text = "\n".join(line.strip() for line in text.splitlines())
    # Collapse multiple blank lines
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

raw_text = extract_text(PDF_PATH)
full_text = clean_text(raw_text)

char_count = len(full_text)
word_count = len(full_text.split())

print(f"Characters : {char_count:,}")
print(f"Words      : {word_count:,}")
print(f"\n--- Preview (first 500 chars) ---\n{full_text[:500]}")

Characters : 366,272
Words      : 55,202

--- Preview (first 500 chars) ---
[PAGE 1] Resolución Directoral Ejecutiva Nº 033-2026-MINEDU/VMGI-PRONABEC Lima, 24 de febrero de 2026 VISTOS: El Informe N° 451-2026-MINEDU/VMGI-PRONABEC-DIBEC-SES, suscrito por la Dirección de Gestión de Becas y la Dirección de Acompañamiento Socioemocional y Bienestar; el Informe N° 042-2026-MINEDU/VMGI-PRONABEC-OPP de la Oficina de Planeamiento y Presupuesto; el Informe N ° 048-2026-MINEDU/VMGI-PRONABEC-OAJ de la Oficina de Asesoría Jurídica, y; CONSIDERANDO: Que, la Ley N° 29837 crea el Prog


## Step 2 — Tokenization & Chunking
Count total tokens with `tiktoken`, justify chunk parameters, and split the text with `RecursiveCharacterTextSplitter`.

In [3]:
import tiktoken
from langchain_text_splitters import RecursiveCharacterTextSplitter

# --- Token count ---
enc = tiktoken.get_encoding("cl100k_base")
total_tokens = len(enc.encode(full_text))
print(f"Total tokens in document: {total_tokens:,}")

# --- Chunk size justification ---
# Embedding limit: 8,192 tokens (gemini-embedding-001)
# chunk_size = 400 tokens  → well within the limit, captures coherent article/paragraph
# chunk_overlap = 60 tokens → ~15% overlap preserves context across chunk boundaries
print("\nChunk size  : 400 tokens  (well within 8,192-token embedding limit)")
print("Chunk overlap: 60 tokens  (~15% overlap to preserve cross-boundary context)")

# --- Splitter ---
CHUNK_SIZE    = 400
CHUNK_OVERLAP = 60

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " "],
    length_function=lambda t: len(enc.encode(t)),
)

metadata_base = {
    "document": "beca18_reglamento",
    "topic": "beca18",
    "language": "es",
}

chunks = splitter.create_documents(
    texts=[full_text],
    metadatas=[metadata_base],
)

avg_len = sum(len(enc.encode(c.page_content)) for c in chunks) / len(chunks)
print(f"\nTotal chunks  : {len(chunks):,}")
print(f"Average length: {avg_len:.1f} tokens")
print(f"\n--- Sample chunk ---\n{chunks[0].page_content[:300]}")

Total tokens in document: 102,824

Chunk size  : 400 tokens  (well within 8,192-token embedding limit)
Chunk overlap: 60 tokens  (~15% overlap to preserve cross-boundary context)

Total chunks  : 321
Average length: 344.0 tokens

--- Sample chunk ---
[PAGE 1] Resolución Directoral Ejecutiva Nº 033-2026-MINEDU/VMGI-PRONABEC Lima, 24 de febrero de 2026 VISTOS: El Informe N° 451-2026-MINEDU/VMGI-PRONABEC-DIBEC-SES, suscrito por la Dirección de Gestión de Becas y la Dirección de Acompañamiento Socioemocional y Bienestar; el Informe N° 042-2026-MINED
